# Les députés représentent-ils leurs électeurs ?

_Projet Python pour la Data Science — 2026_

## Introduction

La démocratie représentative repose sur l'idée que les élus portent la voix et les préoccupations de l'ensemble des citoyens. En science politique, on distingue classiquement la **représentation substantielle** (défendre les intérêts et les idées des électeurs) de la **représentation descriptive** (ressembler socio-démographiquement à la population représentée en termes d'âge, de genre ou de catégorie socioprofessionnelle).

Or, le débat public français souligne régulièrement un décalage croissant entre le profil sociologique des parlementaires de l'Assemblée nationale et la réalité démographique et économique des Français. L'Assemblée nationale est-elle le reflet fidèle du pays, ou constitue-t-elle un espace socialement sélectif ? Et plus finement : les députés ressemblent-ils au moins aux électeurs de **leur propre circonscription** ?

Pour répondre à cette problématique, nous croisons les données individuelles des députés actifs de la XVIIe législature avec les indicateurs socio-économiques de l'INSEE mesurés à l'échelle fine des 577 circonscriptions législatives. Nous structurons notre démarche en trois temps : la construction et l'audit d'une base de données reproductible, une exploration descriptive et cartographique des écarts de représentation, puis une modélisation statistique visant à identifier les déterminants de cette représentativité.

### Sommaire

- **Introduction**
- **I. Création et exploration de la base de données principale**
    - A. Récupération et gel des données sources
    - B. Nettoyage, harmonisation des clés et jointure des tables
    - C. Contrôle de la qualité des données (Audit)
- **II. Visualisations, cartographie et statistiques descriptives**
    - A. Exploration univariée : qui sont les députés ? Qui sont les électeurs ?
    - B. Analyse croisée : le député face à sa circonscription
- **III. Modélisation : les caractéristiques d'une circonscription expliquent-elles le profil de son député ?**
    - A. Objectif et construction des variables cibles
    - B. Modèle 1 : Probabilité d'élire une femme (Régression Logistique)
    - C. Modèle 2 : Déterminants de la CSP et de l'âge du député (Logit & OLS)
    - D. Limites méthodologiques
- **Conclusion**

In [ ]:
# %pip install -r requirements.txt

## I. Création et exploration de la base de données principale

Dans cette première partie, nous construisons un pipeline reproductible permettant d'importer, de figer (_geler_) et d'assembler les différentes sources de données publiques.

| Nom de la table | Description | Source | Mode d'extraction |
| :--- | :--- | :--- | :--- |
| `deputes` | Informations individuelles des députés actifs (âge, civilité, profession, groupe politique, scores parlementaires). | API data.gouv.fr / Datan | API REST (CSV gelé) |
| `insee_indic` | Indicateurs démographiques et économiques par circonscription législative (structure par âge, emploi, CSP, niveau de vie). | INSEE (Recensement & Filosofi) | Fichier Excel (XLSX gelé) |
| `insee_compo` | Table de passage entre communes et circonscriptions législatives. | INSEE | Fichier Excel (XLSX gelé) |
| `contours` | Géométries des 577 circonscriptions législatives françaises (métropole et outre-mer). | data.gouv.fr (J. Desboeufs) | GeoJSON gelé |

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf

from cartes import (
    ajouter_code_circo,
    carte_csp,
    carte_genre,
    carte_groupes,
    carte_indicateur,
    preparer_carte,
)
from collecte_contours import charger_contours, gel_contours
from descriptif import (
    graphique_deputes,
    graphique_insee,
    preparer_deputes,
    tableau_groupes,
    taux_chomage,
    valeurs_manquantes,
)
from extraction_donnees import (
    charger_donnees,
    gel_bases_clean,
    gel_donnees_deputes,
    gel_donnees_insee,
)
from jointures import joindre_deputes_insee
from professions import ajouter_categorie_job

### A. Récupération et gel des données sources

Afin de garantir la stricte reproductibilité de nos résultats — la liste des députés actifs évoluant au gré des démissions, nominations gouvernementales ou élections partielles — nous téléchargeons et sauvegardons localement les fichiers bruts dans `data/data_brut` avant de générer des tables propres au format `.parquet` dans `data/data_clean`.

In [ ]:
# 1. Téléchargement et gel des données brutes (ne retélécharge pas si déjà présent)
gel_donnees_insee(forcer=False)
gel_donnees_deputes(forcer=False)
gel_contours(forcer=False)

# 2. Nettoyage séparé et sauvegarde en format Parquet
gel_bases_clean()

# 3. Chargement des bases propres
deputes, insee_indic, insee_compo, dico_insee = charger_donnees()

### B. Nettoyage, harmonisation des clés et jointure des tables

Le principal défi technique de l'assemblage réside dans l'harmonisation de l'identifiant unique de circonscription (`code_circo`) au format standardisé de l'INSEE sur 5 caractères :
- **France métropolitaine** : code département sur 2 caractères (`01` à `95`, incluant `2A` et `2B` en majuscules pour la Corse) + numéro de circonscription sur 3 chiffres (`001`, `018`, etc.).
- **Outre-mer** : code département sur 3 chiffres (`971` à `976`) + numéro de circonscription sur 2 chiffres (`01`, `02`, etc.).
- **Cas particulier des collectivités d'outre-mer** : l'INSEE regroupe Saint-Barthélemy (`97701`) et Saint-Martin (`97801`) sous un même code statistique (`97801`).
- **Français établis hors de France** : les 11 circonscriptions de l'étranger (`99` / `ZZ`) n'ont pas d'équivalent territorial dans le recensement INSEE métropolitain/DROM.

In [ ]:
deputes_prep = ajouter_categorie_job(preparer_deputes(deputes))
df_complet = joindre_deputes_insee(deputes_prep, insee_indic)

df_complet['taux_chomage'] = taux_chomage(df_complet)
df_complet['part_femmes_pop'] = (
    (df_complet['actemp_hom'] - df_complet['actemp'])
    / (df_complet['actemp_hom'] - df_complet['actemp_fem'])
    * 100
)

display(
    df_complet[[
        'code_circo', 'nom', 'prenom', 'genre', 'age',
        'csp_depute', 'groupeAbrev', 'act_cad', 'nivvie_median_diff'
    ]].head(10)
)

### C. Contrôle de la qualité des données (Audit)

In [ ]:
print('Dimensions de la table jointe :', df_complet.shape)
print('Circonscriptions avec données INSEE :', df_complet['indicateur_present'].sum())
print('Circonscriptions avec un député actif :', df_complet['id'].notna().sum())
print('Doublons sur code_circo :', df_complet['code_circo'].duplicated().sum())
display(valeurs_manquantes(df_complet).head(15))

**Interprétation de l'audit :**
- **Circonscriptions sans député actif** : quelques circonscriptions affichent des valeurs manquantes sur les variables parlementaires (sièges vacants suite à des nominations au gouvernement avant l'entrée en fonction du suppléant, démissions ou invalidations). Elles apparaîtront en gris (« Pas de député ») sur nos cartes.
- **Secret statistique INSEE** : certaines variables économiques (`nivvie_median_diff`) sont masquées (`NaN`) pour les très petites circonscriptions comme Saint-Pierre-et-Miquelon (`97501`) ou Saint-Martin/Saint-Barthélemy (`97801`).

## II. Visualisations, cartographie et statistiques descriptives

### A. Exploration univariée : qui sont les députés ? Qui sont les électeurs ?

In [ ]:
display(tableau_groupes(deputes_prep))

fig_dep = graphique_deputes(deputes_prep)
plt.show()

fig_insee = graphique_insee(df_complet)
plt.show()

### B. Analyse croisée : le député face à sa circonscription

In [ ]:
carte = preparer_carte()

#### 1. Le genre : parité nationale et ancrage territorial

In [ ]:
fig_genre = carte_genre(carte)
plt.show()

fig_part_femmes = carte_indicateur(
    carte,
    'part_femmes_pop',
    'Part de femmes dans la population active occupée (%), par circonscription',
    cmap='Purples',
)
plt.show()

**Analyse comparative :** Alors que les femmes représentent systématiquement environ 50 % à 53 % de la population des circonscriptions, la carte du genre des députés reste majoritairement colorée en orange (hommes), avec une concentration plus forte de femmes députées en Île-de-France, en Bretagne et dans les grandes aires urbaines.

#### 2. La catégorie socioprofessionnelle (CSP) : la surreprésentation des cadres

In [ ]:
part_csp_deputes = (df_complet['csp_depute'].value_counts(normalize=True) * 100).round(1)
print('Répartition des CSP parmi les députés (%) :')
display(part_csp_deputes)
print(f"\nPart moyenne des cadres parmi les actifs en France (INSEE) : {df_complet['act_cad'].mean():.1f} %")

fig_csp = carte_csp(carte)
plt.show()

fig_cadres = carte_indicateur(
    carte, 'act_cad', 'Part de cadres parmi les actifs (%), par circonscription'
)
plt.show()

**Analyse comparative :**
1. **Un monopole des catégories supérieures :** Plus des deux tiers des députés appartiennent à la catégorie des _Cadres et professions intellectuelles supérieures_, alors qu'ils ne représentent qu'environ 18 % à 20 % des actifs en France.
2. **L'invisibilité des classes populaires :** Les ouvriers et employés, qui forment près de la moitié de la population active française, représentent moins de 5 % des députés.
3. **Une géographie duale :** La carte de l'INSEE montre que les cadres sont concentrés à Paris, dans la petite couronne et dans quelques métropoles régionales. À l'inverse, la carte des députés est uniformément bleue (_Cadres_) sur l'ensemble du territoire rural et périurbain.

#### 3. L'âge et le niveau de vie : un décalage générationnel et économique

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(df_complet['age_moyen'], df_complet['age'], alpha=0.5, color='#1f78b4')
axes[0].plot([30, 50], [30, 50], color='crimson', linestyle='--', label='Égalité parfaite (y = x)')
axes[0].set_xlabel('Âge moyen de la population de la circonscription (ans)')
axes[0].set_ylabel('Âge du député (ans)')
axes[0].set_title('Âge du député vs Âge moyen de sa circonscription')
axes[0].legend()

df_complet.boxplot(column='age', by='csp_depute', ax=axes[1], rot=45)
axes[1].set_title("Distribution de l'âge des députés par CSP")
axes[1].set_xlabel('CSP du député')
axes[1].set_ylabel('Âge (ans)')
plt.suptitle('')

plt.tight_layout()
plt.show()

## III. Modélisation : les caractéristiques d'une circonscription expliquent-elles le profil de son député ?

### A. Objectif et construction des variables cibles

In [ ]:
df_model = df_complet.dropna(
    subset=['id', 'genre', 'age', 'csp_depute', 'act_cad', 'nivvie_median_diff', 'taux_chomage']
).copy()

df_model['est_femme'] = (df_model['genre'] == 'Femme').astype(int)
df_model['est_cadre'] = (df_model['csp_depute'] == 'cad').astype(int)
print('Nombre d\'observations retenues pour la modélisation :', len(df_model))

### B. Modèle 1 : Probabilité d'élire une femme (Régression Logistique)

In [ ]:
logit_socio = smf.logit('est_femme ~ act_cad + taux_chomage + age_moyen', data=df_model).fit(disp=False)
print('=== MODÈLE 1A : Déterminants territoriaux de l\'élection d\'une femme ===')
print(logit_socio.summary())

logit_complet = smf.logit(
    'est_femme ~ act_cad + taux_chomage + age_moyen + C(groupeAbrev)', data=df_model
).fit(disp=False)
print('\n=== MODÈLE 1B : Avec contrôle par groupe politique ===')
print(logit_complet.summary())

### C. Modèle 2 : Déterminants de la CSP et de l'âge du député (Logit & OLS)

In [ ]:
logit_cadre = smf.logit('est_cadre ~ act_cad + nivvie_median_diff + taux_chomage', data=df_model).fit(disp=False)
print('=== MODÈLE 2A : Probabilité d\'élire un député Cadre ===')
print(logit_cadre.summary())

ols_age = smf.ols(
    'age ~ age_moyen + nivvie_median_diff + taux_chomage + C(groupeAbrev)', data=df_model
).fit(cov_type='HC3')
print('\n=== MODÈLE 2B : Déterminants de l\'âge du député (OLS) ===')
print(ols_age.summary())

### D. Limites méthodologiques

1. **Biais d'offre politique (sélection avant le vote) :** Nous n'observons que le député _élu_, et non l'ensemble des _candidats_ présents au premier tour.
2. **Limite de la nomenclature des professions :** La profession déclarée correspond parfois à une activité exercée avant une longue carrière d'élu local.
3. **Représentation descriptive vs substantielle :** L'absence de similitude sociologique n'implique pas mécaniquement une divergence de défense des intérêts locaux.

## Conclusion

À la question _« Les députés représentent-ils leurs électeurs ? »_, notre croisement des données de l'Assemblée nationale et de l'INSEE montre que **l'Assemblée nationale ne reflète pas la structure socio-démographique de la population française, ni celle des circonscriptions locales** :

1. **Un biais social massif :** Les cadres et professions intellectuelles supérieures occupent plus des deux tiers des sièges de l'Assemblée nationale.
2. **Un déficit persistant de parité et de jeunesse :** Les hommes et les 45-60 ans demeurent largement surreprésentés.
3. **Le poids des structures partisanes :** Le profil de l'élu dépend beaucoup moins de la sociologie de sa circonscription que des logiques d'investiture des groupes politiques.